In [1]:
import numpy as np

def CRRparams(T, r, v, N):
    # Cox-Ross-Rubinstein (CRR) parameters for binomial tree
    dt = T / N  # time step
    R = np.exp(r * dt)  # discount factor per step
    up = np.exp(v * np.sqrt(dt))  # up-factor
    pu = (R - 1 / up) / (up - 1 / up)  # up probability
    return pu, up, R

def NRTCRR(S0, up, down, N):
    # Create binomial tree for stock prices
    Sbar = np.zeros((N + 1, N + 1))
    Sbar[0, 0] = S0
    for i in range(1, N + 1):
        Sbar[i, 0] = Sbar[i - 1, 0] * down
        for j in range(1, i + 1):
            Sbar[i, j] = Sbar[i - 1, j - 1] * up
    return Sbar

def NRTpsums(Sbar, N):
    # Calculate all partial sums for average calculation
    Ubar = np.zeros((2**N,))
    for m in range(2**N):
        sum_val = 0
        for i in range(N + 1):
            if m & (1 << i):
                sum_val += Sbar[N, i]
        Ubar[m] = sum_val
    return Ubar

def PathAD(pu, R, N):
    # Calculate Arrow-Debreu security prices
    Lbar = np.ones((2**N,))
    for i in range(N):
        for j in range(2**i):
            Lbar[2 * j] *= pu
            Lbar[2 * j + 1] *= (1 - pu)
    return Lbar / (R**N)

def CRRfltAD(T, S0, r, v, N):
    # Implement floating strike option pricing using Arrow-Debreu securities
    pu, up, R = CRRparams(T, r, v, N)
    down = 1 / up
    Sbar = NRTCRR(S0, up, down, N)  # Expanded S tree
    Ubar = NRTpsums(Sbar, N)  # All partial sums
    Lbar = PathAD(pu, R, N)  # Arrow-Debreu security prices
    
    # Corrected mN indexing for N+1 nodes at the final layer
    mN = np.arange(2**N)  # All indices up to the end of Ubar and Lbar
    AvgN = Ubar[mN] / (N + 1)  # Averages at expiry
    
    # Calculate Call and Put premiums
    Sbar_last = np.array([Sbar[N, i] for i in range(N + 1)])  # S values at final time step
    C0 = np.dot(np.maximum(0, Sbar_last - AvgN), Lbar)  # Call
    P0 = np.dot(np.maximum(0, AvgN - Sbar_last), Lbar)  # Put
    return C0, P0

# Testing the function
C0, P0 = CRRfltAD(1, 90, 0.02, 0.20, 4)
print(f"Floating Strike Call Option Price: {C0:.4f}")
print(f"Floating Strike Put Option Price: {P0:.4f}")


ValueError: operands could not be broadcast together with shapes (5,) (16,) 